# Velocity Model Building & Travel Times

This notebook demonstrates **`swe.velocity`** for:

1. Reading and building 1-D and 3-D velocity models
2. Computing travel-time grids
3. Visualizing velocity profiles and travel-time slices

**Dataset**: Published velocity profiles (Hadley & Kanamori, 1977 - Southern California).

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import swe.velocity

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

%matplotlib inline

## Configuration

In [ ]:
# Southern California 1-D velocity model (Hadley & Kanamori, 1977)
# Columns: depth_km, vp_km_s, vs_km_s
VELOCITY_1D = np.array([
    [0.0,  5.5,  3.18],
    [5.5,  6.3,  3.64],
    [16.0, 6.7,  3.87],
    [32.0, 7.8,  4.50],
    [60.0, 8.1,  4.68],
])

# Grid parameters for 3-D model and travel times
NX, NY, NZ = 50, 50, 20
DX = 1.0  # km spacing
ORIGIN = (0.0, 0.0, 0.0)

DATA_DIR = Path("data/velocity_demo")
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Create a 1-D Velocity Model

Build a `VelocityModel` from layer depths and velocities.

In [ ]:
# Write the 1-D model to a file
model_file = DATA_DIR / "socal_1d.csv"
np.savetxt(
    model_file, VELOCITY_1D,
    header="depth_km,vp_km_s,vs_km_s",
    delimiter=",", comments="",
)

# Read back
vmodel = swe.velocity.read_velocity_model_1d(model_file)
print(f"Velocity model: {vmodel}")
print(f"Layers: {len(vmodel.depths)}")
for d, vp, vs in zip(vmodel.depths, vmodel.vp, vmodel.vs):
    print(f"  {d:6.1f} km: Vp={vp:.2f} km/s, Vs={vs:.2f} km/s")

## 2. Visualize Velocity Profile

In [ ]:
fig = swe.velocity.plot_velocity_profile(vmodel)
fig.set_size_inches(8, 6)
plt.tight_layout()
plt.show()

## 3. Build a 3-D Velocity Model

Extrapolate the 1-D profile to a 3-D grid.

In [ ]:
vmodel_3d = swe.velocity.build_velocity_model_3d(
    vmodel,
    nx=NX, ny=NY, nz=NZ,
    dx=DX,
)
print(f"3-D model grid shape: {vmodel_3d.vp_grid.shape}")
print(f"Grid spacing: {DX} km")
print(f"Grid extent: {NX*DX} x {NY*DX} x {NZ*DX} km")

In [ ]:
# Horizontal slice at 10 km depth
fig = swe.velocity.plot_velocity_slice(
    vmodel_3d, depth_km=10.0, component="vp",
)
fig.set_size_inches(8, 7)
plt.tight_layout()
plt.show()

## 4. Compute Travel-Time Grid

Compute P-wave travel times from a source point to all grid nodes.

In [ ]:
# Source at center of model, 8 km depth
source = (NX * DX / 2, NY * DX / 2, 8.0)

tt_grid = swe.velocity.compute_travel_time_grid(
    vmodel_3d, source=source,
)
print(f"Travel-time grid shape: {tt_grid.times.shape}")
print(f"Source: {source}")
print(f"Min travel time: {tt_grid.times.min():.3f} s")
print(f"Max travel time: {tt_grid.times.max():.3f} s")

In [ ]:
# Visualize travel-time slice at source depth
fig = swe.velocity.plot_travel_time_slice(
    tt_grid, depth_km=source[2],
)
fig.set_size_inches(8, 7)
plt.tight_layout()
plt.show()

## 5. Save and Load Travel-Time Grid

In [ ]:
# Save
tt_path = DATA_DIR / "tt_grid.npz"
swe.velocity.save_travel_time_grid(tt_grid, tt_path)
print(f"Saved travel-time grid to {tt_path}")

# Load
tt_loaded = swe.velocity.load_travel_time_grid(tt_path)
print(f"Loaded grid shape: {tt_loaded.times.shape}")
np.testing.assert_allclose(tt_grid.times, tt_loaded.times)
print("Round-trip verified!")

## Summary

| Function | Purpose |
|----------|--------|
| `read_velocity_model_1d()` | Load 1-D depth-velocity profile |
| `build_velocity_model_3d()` | Extrapolate 1-D to 3-D grid |
| `compute_travel_time_grid()` | Eikonal solver for travel times |
| `save_travel_time_grid()` | Serialize to NPZ |
| `plot_velocity_profile()` | Depth vs velocity plot |
| `plot_velocity_slice()` | Horizontal velocity map |
| `plot_travel_time_slice()` | Travel-time contour map |

The velocity module supports the optional `ttcrpy` backend for finite-difference
eikonal solving. Install with `pip install ttcrpy` for 3-D heterogeneous models.